# Welcome to Modal notebooks!

Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [1]:
%uv pip install -q peft accelerate transformers huggingface_hub hf_transfer safetensors

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
# Get the Hugging Face token from Colab secrets
from huggingface_hub import login

# Access your secret (replace 'HF_TOKEN' with your actual secret name)
hf_token = os.environ["HF_TOKEN"]

# Login to Hugging Face
login(token=hf_token)

print("✅ Successfully authenticated with Hugging Face!")

base_model_id = "meta-llama/Llama-3.2-1B"
cpt_adapter_id = "isji/sinllama-1b-cpt"
qa_adapter_id = "isji/sinllama-1b-qa-v5"
tokenizer_id = "polyglots/Extended-Sinhala-LLaMA"
merged_repo_id = "isji/sinllama-1b-qa-v5-merged"

print("Loading Sinhala tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.resize_token_embeddings(len(tokenizer))

print("Merging CPT adapter...")
model = PeftModel.from_pretrained(model, cpt_adapter_id)
model = model.merge_and_unload()

print("Merging QA adapter...")
model = PeftModel.from_pretrained(model, qa_adapter_id)
model = model.merge_and_unload()

model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

print("Merged model ready.")
print(f"Target repo: {merged_repo_id}")

✅ Successfully authenticated with Hugging Face!
Loading Sinhala tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Loading base model...


...

Merging QA adapter...


/usr/local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Merged model ready.
Target repo: isji/sinllama-1b-qa-v5-merged


In [3]:
# This pushes a full merged model, not just a LoRA adapter.
# Use private=True if the repo should not be public.
model.push_to_hub(
    merged_repo_id,
    safe_serialization=True,
    max_shard_size="5GB",
    private=True,
)
tokenizer.push_to_hub(
    merged_repo_id, private=True)

print(f"Merged model pushed to https://huggingface.co/{merged_repo_id}")

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpq0ipj7i2/model.safetensors    :   1%|1         | 39.3MB / 3.09GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpb087lwzw/tokenizer.json       :  90%|######### | 17.5MB / 19.3MB            

Merged model pushed to https://huggingface.co/isji/sinllama-1b-qa-v5-merged
